In [1]:
import numpy as np
import pandas as pd

# Tabel 1: Target & PIC per cabang kota (tabel referensi, dibuat langsung sebagai DataFrame)
data_target_cabang = {
    "kota": ["Magelang", "Yogyakarta", "Semarang", "Solo", "Purworejo"],
    "target_bulanan": [45000000, 60000000, 55000000, 40000000, 30000000],
    "pic_cabang": ["Rani", "Joko", "Sari", "Bayu", "Fitri"],
}

# Tabel 2: Data transaksi (disimpan sebagai CSV, lalu diunggah ke HDFS)
np.random.seed(55)
n = 500
kategori_list = ["Elektronik", "Fashion", "Makanan & Minuman", "Kesehatan & Kecantikan", "Rumah Tangga"]
kota_list = ["Magelang", "Yogyakarta", "Semarang", "Solo", "Purworejo"]
data_transaksi_t5 = {
    "order_id": [f"TRX-{i}" for i in range(n)],
    "kategori": np.random.choice(kategori_list, size=n),
    "kota": np.random.choice(kota_list, size=n),
    "unit_terjual": np.random.randint(1, 10, size=n),
    "harga_satuan": np.random.choice([25000, 50000, 75000, 100000, 150000], size=n),
}
pd.DataFrame(data_transaksi_t5).to_csv("transaksi_tugas5.csv", index=False)

!hdfs dfs -mkdir -p /user/nabill/tugas5
!hdfs dfs -put -f transaksi_tugas5.csv /user/nabill/tugas5/
print("Dataset siap. Tabel transaksi sudah diunggah ke HDFS: /user/nabill/tugas5/transaksi_tugas5.csv")
print("Simpan juga 'data_target_cabang' di atas — kalian akan membuatnya menjadi DataFrame sendiri di notebook tugas.")

Dataset siap. Tabel transaksi sudah diunggah ke HDFS: /user/nabill/tugas5/transaksi_tugas5.csv
Simpan juga 'data_target_cabang' di atas — kalian akan membuatnya menjadi DataFrame sendiri di notebook tugas.


In [3]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum as spark_sum, avg, count, rank, row_number, when
from pyspark.sql.window import Window

spark = SparkSession.builder \
    .appName("Pertemuan5-JoinWindowSQL") \
    .master("local[*]") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

print("SparkSession siap. Versi Spark:", spark.version)

SparkSession siap. Versi Spark: 3.5.9


In [4]:
df = spark.read.csv(
    "hdfs://localhost:9000/user/nabill/tugas5/data_transaksi_ecommerce.csv",
    header=True, inferSchema=True
)
df.printSchema()
print("Jumlah baris:", df.count())
df.show(10)

root
 |-- order_id: string (nullable = true)
 |-- tanggal: timestamp (nullable = true)
 |-- kategori: string (nullable = true)
 |-- kota: string (nullable = true)
 |-- unit_terjual: integer (nullable = true)
 |-- harga_satuan: integer (nullable = true)
 |-- metode_pembayaran: string (nullable = true)

Jumlah baris: 600
+--------+-------------------+--------------------+----------+------------+------------+-----------------+
|order_id|            tanggal|            kategori|      kota|unit_terjual|harga_satuan|metode_pembayaran|
+--------+-------------------+--------------------+----------+------------+------------+-----------------+
|ORD-1000|2026-07-07 00:00:00|Kesehatan & Kecan...|  Magelang|           9|      250000|         E-Wallet|
|ORD-1001|2026-07-20 00:00:00|   Makanan & Minuman|Yogyakarta|           9|       50000|     Kartu Kredit|
|ORD-1002|2026-07-29 00:00:00|             Fashion| Purworejo|           2|       50000|     Kartu Kredit|
|ORD-1003|2026-07-15 00:00:00|       

In [5]:
import pandas as pd
df_target = spark.createDataFrame(pd.DataFrame(data_target_cabang))
df = df.withColumn("total_pendapatan", col("unit_terjual") * col("harga_satuan"))

ringkasan = df.groupBy("kota").agg(
    spark_sum("total_pendapatan").alias("total_pendapatan")
)

hasil = ringkasan.join(df_target, on="kota", how="inner")
hasil = hasil.withColumn(
    "pencapaian_persen", (col("total_pendapatan") / col("target_bulanan") * 100)
)

hasil.orderBy(col("pencapaian_persen").desc()).show(10)

+----------+----------------+--------------+----------+------------------+
|      kota|total_pendapatan|target_bulanan|pic_cabang| pencapaian_persen|
+----------+----------------+--------------+----------+------------------+
| Purworejo|       104675000|      30000000|     Fitri| 348.9166666666667|
|  Magelang|       109925000|      45000000|      Rani|244.27777777777777|
|      Solo|        83950000|      40000000|      Bayu|           209.875|
|Yogyakarta|       104625000|      60000000|      Joko|           174.375|
|  Semarang|        87200000|      55000000|      Sari|158.54545454545453|
+----------+----------------+--------------+----------+------------------+



In [6]:
ringkasan_kota_kategori = df.groupBy("kota", "kategori").agg(
    spark_sum("total_pendapatan").alias("total_pendapatan")
)

window_b = Window.partitionBy("kota").orderBy(col("total_pendapatan").desc())

hasil_b = ringkasan_kota_kategori.withColumn("no_urut", row_number().over(window_b)) \
    .filter(col("no_urut") == 1) \
    .orderBy("kota")

hasil_b.show(10)

+----------+-----------------+----------------+-------+
|      kota|         kategori|total_pendapatan|no_urut|
+----------+-----------------+----------------+-------+
|  Magelang|Makanan & Minuman|        25625000|      1|
| Purworejo|          Fashion|        33850000|      1|
|  Semarang|       Elektronik|        30575000|      1|
|      Solo|          Fashion|        24925000|      1|
|Yogyakarta|          Fashion|        27625000|      1|
+----------+-----------------+----------------+-------+



In [7]:
df.createOrReplaceTempView("transaksi")
df_target.createOrReplaceTempView("target")

hasil_c = spark.sql('''
    SELECT t.kota, tg.pic_cabang, COUNT(*) AS jumlah_transaksi
    FROM transaksi t
    JOIN target tg ON t.kota = tg.kota
    GROUP BY t.kota, tg.pic_cabang
    ORDER BY jumlah_transaksi DESC 
''')
hasil_c.show(10)

+----------+----------+----------------+
|      kota|pic_cabang|jumlah_transaksi|
+----------+----------+----------------+
|Yogyakarta|      Joko|             129|
| Purworejo|     Fitri|             122|
|  Magelang|      Rani|             119|
|  Semarang|      Sari|             118|
|      Solo|      Bayu|             112|
+----------+----------+----------------+

